# 5. Visualise PyNNLF Output: AEDP Aggregation Levels

Visualises the 1-day-ahead AEDP aggregation-level experiment after notebooks 3 and 4 have been run. This version assumes 3 samples per aggregation level.


## 1. Setup And Paths

In [1]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results" / "03_aedp_aggregation_level"
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ORDER = ["m1_naive_hp1", "m6_lr_hp1", "m17_xgb_hp1"]
MODEL_LABELS = {"m1_naive_hp1": "Naive", "m6_lr_hp1": "Linear regression", "m17_xgb_hp1": "XGBoost"}
COLORS = {"m1_naive_hp1": "#2f5597", "m6_lr_hp1": "#70ad47", "m17_xgb_hp1": "#c55a11"}
SAMPLES_PER_LEVEL = 3
AGGREGATION_LEVELS = [1, 10, 100, 1000]
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "axes.grid": True, "grid.alpha": 0.25})
def save_figure(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")

Publication project: <local path redacted>
Repository root: <local path redacted>


## 2. Load Processed Tables

In [2]:
required = {"recap": RESULTS_DIR / "aedp_aggregation_fh8_recap.csv", "summary": RESULTS_DIR / "aedp_aggregation_fh8_summary_by_level_model.csv"}
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing processed result files. Run notebook 4 first. " + str(missing))
recap = pd.read_csv(required["recap"])
summary = pd.read_csv(required["summary"])
display(summary.round(3))

    aggregation_level_hh    model_name  ...  mean_runtime_s  n_samples
0                      1  m1_naive_hp1  ...           0.002          3
1                      1     m6_lr_hp1  ...           0.817          3
2                      1   m17_xgb_hp1  ...          30.234          3
3                     10  m1_naive_hp1  ...           0.001          3
4                     10     m6_lr_hp1  ...           0.703          3
5                     10   m17_xgb_hp1  ...          26.570          3
6                    100  m1_naive_hp1  ...           0.002          3
7                    100     m6_lr_hp1  ...           0.616          3
8                    100   m17_xgb_hp1  ...          24.117          3
9                   1000  m1_naive_hp1  ...           0.002          3
10                  1000     m6_lr_hp1  ...           0.868          3
11                  1000   m17_xgb_hp1  ...          28.482          3

[12 rows x 7 columns]


## 3. nRMSE And Normalisation Denominator

This combined figure places the three model-wise nRMSE panels beside the positive peak net-load-per-household panel used in nRMSE normalisation.

In [3]:
from matplotlib.ticker import MaxNLocator

dataset_rows = (
    recap[["dataset_id", "filename", "aggregation_level_hh", "sample_no", "unique_households", "total_household_weight"]]
    .drop_duplicates("dataset_id")
    .sort_values(["aggregation_level_hh", "sample_no"])
)

peak_rows = []
for row in dataset_rows.itertuples(index=False):
    path = DATA_DIR / row.filename
    if not path.exists():
        raise FileNotFoundError(f"Missing aggregation dataset for peak-load calculation: {path}")
    series = pd.read_csv(path, usecols=["netload_kW"])["netload_kW"]
    if series.isna().any():
        raise ValueError(f"Missing netload_kW values in {path.name}")
    weight = float(row.total_household_weight)
    peak_rows.append({
        "dataset_id": row.dataset_id,
        "aggregation_level_hh": int(row.aggregation_level_hh),
        "sample_no": int(row.sample_no),
        "unique_households": int(row.unique_households),
        "total_household_weight": weight,
        "positive_peak_kW": float(series.max()),
        "positive_peak_kW_per_household": float(series.max()) / weight,
        "minimum_netload_kW": float(series.min()),
        "minimum_netload_kW_per_household": float(series.min()) / weight,
    })
peak_stats = pd.DataFrame(peak_rows)
peak_summary = (
    peak_stats
    .groupby("aggregation_level_hh", as_index=False)
    .agg(
        mean_positive_peak_kW_per_household=("positive_peak_kW_per_household", "mean"),
        sample_std_positive_peak_kW_per_household=("positive_peak_kW_per_household", "std"),
        n_samples=("dataset_id", "nunique"),
    )
)

figure7_box_style = {
    "showmeans": True,
    "patch_artist": True,
    "meanprops": {"marker": "^", "markerfacecolor": "#2E9E45", "markeredgecolor": "#2E9E45", "markersize": 13},
    "medianprops": {"color": "#D55E00", "linewidth": 1.4},
    "boxprops": {"facecolor": "#F4F6F8", "edgecolor": "#2F4D67", "linewidth": 1.2},
    "whiskerprops": {"color": "#2F4D67", "linewidth": 1.2},
    "capprops": {"color": "#2F4D67", "linewidth": 1.2},
    "flierprops": {"marker": "o", "markerfacecolor": "#EB932C", "markeredgecolor": "white", "markersize": 9, "alpha": 0.9},
}

nrmse_values = pd.to_numeric(recap["test_nRMSE"], errors="coerce").dropna()
y_pad = max((nrmse_values.max() - nrmse_values.min()) * 0.08, 0.5)
shared_nrmse_ylim = (nrmse_values.min() - y_pad, nrmse_values.max() + y_pad)

nrmse_data_by_model = {
    model: [
        recap.loc[(recap["model_name"].eq(model)) & (recap["aggregation_level_hh"].eq(level)), "test_nRMSE"].dropna().to_numpy()
        for level in AGGREGATION_LEVELS
    ]
    for model in MODEL_ORDER
}
peak_data = [
    peak_stats.loc[peak_stats["aggregation_level_hh"].eq(level), "positive_peak_kW_per_household"].dropna().to_numpy()
    for level in AGGREGATION_LEVELS
]

def boxplot_with_tick_labels(ax, data, labels, **kwargs):
    try:
        return ax.boxplot(data, tick_labels=labels, **kwargs)
    except TypeError:
        return ax.boxplot(data, labels=labels, **kwargs)

def style_axis(ax, *, title, xlabel="Households", ylabel=None, fontsize=21, title_size=24, nbins=4):
    ax.set_title(title, fontsize=title_size, pad=10)
    ax.set_xlabel(xlabel, fontsize=fontsize, labelpad=8)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=fontsize, labelpad=10)
    ax.tick_params(axis="both", labelsize=fontsize)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.grid(axis="y", alpha=0.25)
    ax.grid(axis="x", visible=False)

def draw_nrmse_panel(ax, model, *, ylabel=False, fontsize=21):
    boxplot_with_tick_labels(
        ax,
        nrmse_data_by_model[model],
        [str(level) for level in AGGREGATION_LEVELS],
        **figure7_box_style,
    )
    ax.set_ylim(*shared_nrmse_ylim)
    style_axis(
        ax,
        title=MODEL_LABELS[model],
        ylabel="Test nRMSE\n(%)" if ylabel else None,
        fontsize=fontsize,
    )

def draw_peak_panel(ax, *, ylabel=True, fontsize=21):
    boxplot_with_tick_labels(
        ax,
        peak_data,
        [str(level) for level in AGGREGATION_LEVELS],
        **figure7_box_style,
    )
    for x_pos, values in enumerate(peak_data, start=1):
        if len(values) == 0:
            continue
        jitter = np.linspace(-0.08, 0.08, len(values)) if len(values) > 1 else np.array([0.0])
        ax.scatter(
            np.full(len(values), x_pos) + jitter,
            values,
            s=72,
            color="#EB932C",
            edgecolor="white",
            linewidth=0.7,
            alpha=0.85,
            zorder=3,
        )
    style_axis(
        ax,
        title="Peak per household",
        ylabel="Positive peak net load\n(kW/household)" if ylabel else None,
        fontsize=fontsize,
    )

fig, axes = plt.subplots(
    1,
    len(MODEL_ORDER) + 1,
    figsize=(21.5, 6.4),
    gridspec_kw={"width_ratios": [1, 1, 1, 1.05], "wspace": 0.36},
)
for ax, model in zip(axes[:len(MODEL_ORDER)], MODEL_ORDER):
    draw_nrmse_panel(ax, model, ylabel=(model == MODEL_ORDER[0]), fontsize=21)
draw_peak_panel(axes[-1], ylabel=True, fontsize=21)
fig.subplots_adjust(left=0.07, right=0.99, bottom=0.19, top=0.86, wspace=0.40)
save_figure(fig, "fig01a_aedp_aggregation_nrmse_and_peak_normalisation_1x4.png")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12.4, 9.6), constrained_layout=True)
draw_nrmse_panel(axes[0, 0], MODEL_ORDER[0], ylabel=True, fontsize=20)
draw_nrmse_panel(axes[0, 1], MODEL_ORDER[1], ylabel=False, fontsize=20)
draw_nrmse_panel(axes[1, 0], MODEL_ORDER[2], ylabel=True, fontsize=20)
draw_peak_panel(axes[1, 1], ylabel=True, fontsize=20)
save_figure(fig, "fig01b_aedp_aggregation_nrmse_and_peak_normalisation_2x2.png")
plt.show()


Saved: <local path redacted>
notebooks\03_aedp_aggregation_level\5_visualise_pynnlf_aedp_aggregation.ipynb:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ]


## 4. RMSE Per Household Distribution

This figure uses the same faceted boxplot style as the nRMSE distribution, but replaces nRMSE with `test_RMSE / total_household_weight`.

In [5]:
from matplotlib.ticker import MaxNLocator

recap = recap.copy()
recap["test_RMSE_per_household"] = pd.to_numeric(recap["test_RMSE"], errors="coerce") / pd.to_numeric(recap["total_household_weight"], errors="coerce")

rmse_per_hh_summary = (
    recap
    .groupby(["aggregation_level_hh", "model_name"], as_index=False)
    .agg(
        mean_test_RMSE_per_household=("test_RMSE_per_household", "mean"),
        sample_std_test_RMSE_per_household=("test_RMSE_per_household", "std"),
        n_samples=("dataset_id", "nunique"),
    )
)
if rmse_per_hh_summary["mean_test_RMSE_per_household"].isna().any():
    raise ValueError("Missing RMSE-per-household values; check test_RMSE and total_household_weight columns.")

figure8_box_style = {
    "showmeans": True,
    "patch_artist": True,
    "meanprops": {"marker": "^", "markerfacecolor": "#2E9E45", "markeredgecolor": "#2E9E45", "markersize": 13},
    "medianprops": {"color": "#D55E00", "linewidth": 1.4},
    "boxprops": {"facecolor": "#F4F6F8", "edgecolor": "#2F4D67", "linewidth": 1.2},
    "whiskerprops": {"color": "#2F4D67", "linewidth": 1.2},
    "capprops": {"color": "#2F4D67", "linewidth": 1.2},
    "flierprops": {"marker": "o", "markerfacecolor": "#EB932C", "markeredgecolor": "white", "markersize": 9, "alpha": 0.9},
}

fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(15.8, 5.9), sharey=True)
for ax, model in zip(axes, MODEL_ORDER):
    data = [
        recap.loc[(recap["model_name"].eq(model)) & (recap["aggregation_level_hh"].eq(level)), "test_RMSE_per_household"].dropna().to_numpy()
        for level in AGGREGATION_LEVELS
    ]
    boxplot_with_tick_labels(ax, data, [str(level) for level in AGGREGATION_LEVELS], **figure8_box_style)
    ax.set_title(MODEL_LABELS[model], fontsize=24, pad=12)
    ax.set_xlabel("Households", fontsize=21, labelpad=10)
    ax.tick_params(axis="both", labelsize=21)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.grid(axis="y", alpha=0.25)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Test RMSE per household\n(kW/household)", fontsize=21, labelpad=12)
fig.suptitle(f"AEDP aggregation-level RMSE per household across {SAMPLES_PER_LEVEL} samples", fontsize=24, y=1.03)
fig.tight_layout()
save_figure(fig, "fig02_aedp_aggregation_rmse_per_household.png")
plt.show()

display(rmse_per_hh_summary.round(4))


Saved: <local path redacted>
notebooks\03_aedp_aggregation_level\5_visualise_pynnlf_aedp_aggregation.ipynb:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "execution": {
    aggregation_level_hh  ... n_samples
0                      1  ...         3
1                      1  ...         3
2                      1  ...         3
3                     10  ...         3
4                     10  ...         3
5                     10  ...         3
6                    100  ...         3
7                    100  ...         3
8                    100  ...         3
9                   1000  ...         3
10                  1000  ...         3
11                  1000  ...         3

[12 rows x 5 columns]


## 5. Raw Test RMSE Distribution

This figure uses the same faceted boxplot style as the nRMSE distribution, but replaces nRMSE with raw aggregate-scale test RMSE. The y-axis uses a log scale because the kW scale grows with aggregation size.

In [6]:
raw_rmse_summary = (
    recap
    .groupby(["aggregation_level_hh", "model_name"], as_index=False)
    .agg(
        mean_test_RMSE=("test_RMSE", "mean"),
        sample_std_test_RMSE=("test_RMSE", "std"),
        n_samples=("dataset_id", "nunique"),
    )
)

fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(14, 4), sharey=True)
for ax, model in zip(axes, MODEL_ORDER):
    data = [recap.loc[(recap["model_name"].eq(model)) & (recap["aggregation_level_hh"].eq(level)), "test_RMSE"].dropna().to_numpy() for level in AGGREGATION_LEVELS]
    ax.boxplot(data, labels=[str(level) for level in AGGREGATION_LEVELS], showmeans=True)
    ax.set_title(MODEL_LABELS[model])
    ax.set_xlabel("Households")
    ax.set_yscale("log")
axes[0].set_ylabel("Test RMSE (kW, log scale)")
fig.suptitle(f"AEDP aggregation-level raw RMSE across {SAMPLES_PER_LEVEL} samples", y=1.03)
fig.tight_layout()
save_figure(fig, "fig03_aedp_aggregation_raw_rmse.png")
plt.show()

display(raw_rmse_summary.round(4))


Saved: <local path redacted>
notebooks\03_aedp_aggregation_level\5_visualise_pynnlf_aedp_aggregation.ipynb:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "cell_type": "code",
    aggregation_level_hh    model_name  ...  sample_std_test_RMSE  n_samples
0                      1   m17_xgb_hp1  ...                0.3490          3
1                      1  m1_naive_hp1  ...                0.5183          3
2                      1     m6_lr_hp1  ...                0.4087          3
3                     10   m17_xgb_hp1  ...                0.9742          3
4                     10  m1_naive_hp1  ...                1.5971          3
5                     10     m6_lr_hp1  ...                1.3735          3
6                    100   m17_xgb_hp1  ...                0.8911          3
7                    100  m1_naive_hp1  ...                1.1260          3
8                    100     m6_lr_hp1  ...                0.9622          3
9                   10

## 6. Peak Net Load Summary

These tables support the fourth panel in Figure 1 and are retained for checking the nRMSE normalisation denominator.

In [7]:
display(peak_summary.round(4))
display(peak_stats.round(4))


Saved: <local path redacted>
notebooks\03_aedp_aggregation_level\5_visualise_pynnlf_aedp_aggregation.ipynb:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "display(summary.round(3))"
   aggregation_level_hh  ...  n_samples
0                     1  ...          3
1                    10  ...          3
2                   100  ...          3
3                  1000  ...          3

[4 rows x 4 columns]
   dataset_id  ...  minimum_netload_kW_per_household
0        ds25  ...                           -7.3635
1        ds26  ...                           -9.7532
2        ds27  ...                           -4.0017
3        ds28  ...                           -2.6625
4        ds29  ...                           -2.6940
5        ds30  ...                           -5.6248
6        ds31  ...                           -3.3263
7        ds32  ...                           -3.4098
8        ds33  ...                           -3.3367
9        ds34  ...               

## 7. Figure Inventory

In [8]:
for path in sorted(FIGURES_DIR.glob("fig*.png")):
    print(path)

<local path redacted>
<local path redacted>
<local path redacted>
<local path redacted>
<local path redacted>
